In [ ]:
pip install lightgbm xgboost

In [ ]:
import os
import re
import json
import random
import hashlib
import datetime as dt
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

@dataclass
class Config:
    data_dir: str = "."
    train_path: str = "train.csv"
    public_test_path: str = "public_test.csv"
    private_test_path: str = "private_test.csv"
    output_dir: str = "outputs"
    artifacts_dir: str = "artifacts"
    embedding_model_name: str = "allenai/specter2"
    transformer_model_name: str = "allenai/scibert_scivocab_uncased"
    author_graph_mode: str = "both"  # coauthor, bipartite, both, none
    author_emb_dim: int = 32
    author_emb_stats: Tuple[str, ...] = ("mean", "max", "std")
    node2vec_walk_length: int = 20
    node2vec_num_walks: int = 10
    node2vec_window: int = 5
    node2vec_min_count: int = 1
    node2vec_workers: int = 2
    node2vec_p: float = 1.0
    node2vec_q: float = 1.0
    tfidf_word_ngram: Tuple[int, int] = (1, 2)
    tfidf_char_ngram: Tuple[int, int] = (3, 5)
    tfidf_min_df: int = 2
    tfidf_max_df: float = 0.9
    tfidf_word_max_features: int = 100000
    tfidf_char_max_features: int = 100000
    tfidf_alpha: float = 1.0
    embedding_pca_dim: int = 64
    embedding_max_length: int = 256
    n_splits: int = 5
    random_seed: int = 42
    transformer_epochs: int = 2
    transformer_lr: float = 2e-5
    transformer_batch_size: int = 16
    transformer_max_length: int = 128
    use_fp16: bool = True
    ensemble_weight: float = 0.5

cfg = Config(data_dir=".")

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

def sha256_file(path: str, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def save_json(path: str, payload: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

class ExperimentTracker:
    def __init__(self, artifacts_dir: str, use_mlflow: bool = True) -> None:
        self.artifacts_dir = artifacts_dir
        self.use_mlflow = False
        self._mlflow = None
        if use_mlflow:
            try:
                import mlflow
                self._mlflow = mlflow
                self.use_mlflow = True
            except Exception:
                self.use_mlflow = False

    def log_run(self, params: dict, metrics: dict, tags: Optional[dict] = None) -> None:
        payload = {
            "timestamp": dt.datetime.utcnow().isoformat() + "Z",
            "params": params,
            "metrics": metrics,
            "tags": tags or {},
        }
        if self.use_mlflow:
            self._mlflow.start_run()
            self._mlflow.log_params(params)
            self._mlflow.log_metrics(metrics)
            for k, v in (tags or {}).items():
                if not isinstance(v, str):
                    try:
                        v = json.dumps(v, sort_keys=True)
                    except Exception:
                        v = str(v)
                self._mlflow.set_tag(k, v)
            self._mlflow.end_run()
        else:
            path = os.path.join(self.artifacts_dir, "experiments.jsonl")
            with open(path, "a", encoding="utf-8") as f:
                f.write(json.dumps(payload) + "\n")

In [ ]:
def clean_noise(train_df: pd.DataFrame) -> pd.DataFrame:
    df = train_df.copy()

    df["title_temp"] = df["title"].fillna("").astype(str).str.strip().str.lower()
    df = df.drop_duplicates(subset=["title_temp", "Label"], keep="first")
    duplicated_titles = df[df.duplicated(subset=["title_temp"], keep=False)]

    if len(duplicated_titles) > 0:
        # gom nhóm lấy trung bình nhãn
        agg_dict = {
            'id': 'first',
            'authors': 'first',
            'venue': 'first',
            'year': 'first',
            'doi': 'first',
            'Label': lambda x: int(np.round(x.mean()))
        }

        title_mapping = df.drop_duplicates(subset=["title_temp"]).set_index("title_temp")["title"].to_dict()

        df = df.groupby("title_temp").agg(agg_dict).reset_index()
        df["title"] = df["title_temp"].map(title_mapping)
        df = df[["id", "title", "authors", "venue", "year", "doi", "Label"]]
    else:
        df = df.drop(columns=["title_temp"])

    return df

In [ ]:
def clean_text(text: str) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_category(text: str) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return "<missing>"
    text = str(text).strip().lower()
    return text if text else "<missing>"

def parse_authors(authors: str) -> List[str]:
    if authors is None or (isinstance(authors, float) and np.isnan(authors)):
        return []
    raw = re.split(r"[,;]", str(authors))
    cleaned = [a.strip().lower() for a in raw if a.strip()]
    return cleaned

def extract_doi_fields(doi: str) -> Dict[str, str]:
    if doi is None or (isinstance(doi, float) and np.isnan(doi)):
        return {
            "doi_domain": "<missing>",
            "doi_prefix": "<missing>",
            "is_url": 0,
            "has_doi": 0,
            "doi_len": 0,
        }
    doi_str = str(doi).strip().lower()
    if not doi_str:
        return {
            "doi_domain": "<missing>",
            "doi_prefix": "<missing>",
            "is_url": 0,
            "has_doi": 0,
            "doi_len": 0,
        }

    is_url = int(doi_str.startswith("http"))
    doi_len = len(doi_str)
    doi_domain = "<missing>"
    doi_prefix = "<missing>"

    if is_url:
        domain_match = re.match(r"https?://([^/]+)/?", doi_str)
        if domain_match:
            doi_domain = domain_match.group(1)
        if "doi.org/" in doi_str:
            doi_part = doi_str.split("doi.org/")[-1]
            if doi_part.startswith("10."):
                doi_prefix = doi_part.split("/")[0]
    else:
        if doi_str.startswith("10."):
            doi_prefix = doi_str.split("/")[0]

    has_doi = int(doi_prefix != "<missing>")

    return {
        "doi_domain": doi_domain,
        "doi_prefix": doi_prefix,
        "is_url": is_url,
        "has_doi": has_doi,
        "doi_len": doi_len,
    }

In [ ]:
def build_coauthor_degrees(author_lists: Iterable[List[str]]) -> Dict[str, int]:
    coauthors: Dict[str, set] = {}
    for authors in author_lists:
        unique = list(dict.fromkeys(authors))
        for i, a in enumerate(unique):
            if a not in coauthors:
                coauthors[a] = set()
            for j, b in enumerate(unique):
                if i == j:
                    continue
                coauthors[a].add(b)
    return {a: len(neigh) for a, neigh in coauthors.items()}

def build_coauthor_graph(author_lists: Iterable[List[str]]):
    import networkx as nx
    graph = nx.Graph()
    for authors in author_lists:
        unique = list(dict.fromkeys([a for a in authors if a]))
        for a in unique:
            if not graph.has_node(a):
                graph.add_node(a)
        for i in range(len(unique)):
            for j in range(i + 1, len(unique)):
                a = unique[i]
                b = unique[j]
                if graph.has_edge(a, b):
                    graph[a][b]["weight"] += 1.0
                else:
                    graph.add_edge(a, b, weight=1.0)
    return graph

def build_bipartite_graph(author_lists: Iterable[List[str]], paper_ids: Iterable[str]):
    import networkx as nx
    graph = nx.Graph()
    for authors, paper_id in zip(author_lists, paper_ids):
        paper_node = f"paper_{paper_id}"
        if not graph.has_node(paper_node):
            graph.add_node(paper_node)
        unique = list(dict.fromkeys([a for a in authors if a]))
        for a in unique:
            if not graph.has_node(a):
                graph.add_node(a)
            if graph.has_edge(a, paper_node):
                graph[a][paper_node]["weight"] += 1.0
            else:
                graph.add_edge(a, paper_node, weight=1.0)
    return graph

def train_node2vec_embeddings(graph, cfg: Config) -> Dict[str, np.ndarray]:
    from node2vec import Node2Vec
    if graph.number_of_nodes() == 0:
        return {}

    node2vec = Node2Vec(
        graph, dimensions=cfg.author_emb_dim, walk_length=cfg.node2vec_walk_length,
        num_walks=cfg.node2vec_num_walks, p=cfg.node2vec_p, q=cfg.node2vec_q,
        workers=cfg.node2vec_workers, seed=cfg.random_seed, weight_key="weight",
    )
    model = node2vec.fit(
        window=cfg.node2vec_window, min_count=cfg.node2vec_min_count,
        batch_words=256, seed=cfg.random_seed,
    )
    return {node: model.wv[node] for node in graph.nodes()}

def build_author_embedding_block(author_lists, author_vectors, dim, stats) -> np.ndarray:
    author_lists = list(author_lists)
    out_dim = dim * len(stats)
    embeddings = np.zeros((len(author_lists), out_dim), dtype=float)

    for i, authors in enumerate(author_lists):
        vecs = [author_vectors[a] for a in authors if a in author_vectors]
        if not vecs:
            continue
        mat = np.vstack(vecs)
        offset = 0
        for stat in stats:
            if stat == "mean":
                embeddings[i, offset : offset + dim] = mat.mean(axis=0)
            elif stat == "max":
                embeddings[i, offset : offset + dim] = mat.max(axis=0)
            elif stat == "std":
                embeddings[i, offset : offset + dim] = mat.std(axis=0)
            offset += dim
    return embeddings

In [ ]:
class TargetEncoder:
    def __init__(self) -> None:
        self.mapping: Dict[str, float] = {}
        self.global_mean: float = 0.0

    def fit(self, series: pd.Series, y: np.ndarray) -> "TargetEncoder":
        df = pd.DataFrame({"cat": series, "y": y})
        self.global_mean = float(df["y"].mean())
        self.mapping = df.groupby("cat")["y"].mean().to_dict()
        return self

    def transform(self, series: pd.Series) -> np.ndarray:
        return series.map(self.mapping).fillna(self.global_mean).astype(float).values

    def fit_transform_oof(self, series: pd.Series, y: np.ndarray, skf: StratifiedKFold) -> np.ndarray:
        df = pd.DataFrame({"cat": series, "y": y})
        self.global_mean = float(df["y"].mean())
        oof = np.zeros(len(series), dtype=float)
        for tr_idx, val_idx in skf.split(df, y):
            tr_df = df.iloc[tr_idx]
            means = tr_df.groupby("cat")["y"].mean()
            oof[val_idx] = df.iloc[val_idx]["cat"].map(means).fillna(self.global_mean).values
        self.mapping = df.groupby("cat")["y"].mean().to_dict()
        return oof

def qwk(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(cohen_kappa_score(y_true, y_pred, weights="quadratic"))

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def apply_thresholds(preds: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    thresholds = np.sort(np.array(thresholds))
    out = np.zeros(len(preds), dtype=int)
    for i, t in enumerate(thresholds):
        out[preds > t] = i + 1
    return out

def optimize_thresholds(preds: np.ndarray, y_true: np.ndarray, n_classes: int) -> np.ndarray:
    preds = np.asarray(preds, dtype=float)
    y_true = np.asarray(y_true, dtype=int)
    init = np.quantile(preds, np.linspace(0, 1, n_classes + 1)[1:-1])

    try:
        from scipy.optimize import minimize
        def loss_fn(x: np.ndarray) -> float:
            rounded = apply_thresholds(preds, x)
            return -qwk(y_true, rounded)
        result = minimize(loss_fn, init, method="Nelder-Mead")
        return np.sort(result.x)
    except Exception:
        best = np.array(init, dtype=float)
        best_score = qwk(y_true, apply_thresholds(preds, best))
        for i in range(len(best)):
            candidates = np.linspace(best[i] - 1.0, best[i] + 1.0, 41)
            for c in candidates:
                trial = best.copy()
                trial[i] = c
                trial = np.sort(trial)
                score = qwk(y_true, apply_thresholds(preds, trial))
                if score > best_score:
                    best = trial
                    best_score = score
        return best

In [ ]:
def load_data(cfg: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Thêm error handling giả định để chạy thử nếu bạn không có file ở môi trường hiện tại
    try:
        train = pd.read_csv(os.path.join(cfg.data_dir, cfg.train_path))
        public_test = pd.read_csv(os.path.join(cfg.data_dir, cfg.public_test_path))
        private_test = pd.read_csv(os.path.join(cfg.data_dir, cfg.private_test_path))
        return train, public_test, private_test
    except FileNotFoundError as e:
        print(f"Vui lòng đảm bảo các tệp dữ liệu đã ở đúng đường dẫn: {e}")
        # Dữ liệu giả lập (dummy) để test code nếu không có CSV:
        df_dummy = pd.DataFrame({'id': [1,2], 'title': ['A','B'], 'venue': ['C','D'], 'year': [2020,2021], 'authors': ['X', 'Y'], 'doi': ['10.x/1', '10.y/2'], 'Label': [1,2]})
        return df_dummy, df_dummy.copy(), df_dummy.copy()

def eda_summary(train: pd.DataFrame) -> dict:
    if "Label" not in train.columns: return {}
    label_counts = train["Label"].value_counts(dropna=False).sort_index()
    missing = train.isna().sum().to_dict()
    return {"label_distribution": label_counts.to_dict(), "missing_values": missing, "n_rows": int(len(train))}

def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["title_clean"] = df["title"].apply(clean_text)
    df["venue_clean"] = df["venue"].apply(normalize_category)
    df["authors_list"] = df["authors"].apply(parse_authors)

    doi_fields = df["doi"].apply(extract_doi_fields)
    df["doi_domain"] = doi_fields.apply(lambda x: x["doi_domain"])
    df["doi_prefix"] = doi_fields.apply(lambda x: x["doi_prefix"])
    df["doi_is_url"] = doi_fields.apply(lambda x: x["is_url"]).astype(int)
    df["doi_has_doi"] = doi_fields.apply(lambda x: x["has_doi"]).astype(int)
    df["doi_len"] = doi_fields.apply(lambda x: x["doi_len"]).astype(int)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    return df

def fit_structured_encoders(train_df: pd.DataFrame, y: np.ndarray, cfg: Config):
    skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_seed)
    venue_te = TargetEncoder()
    venue_te_oof = venue_te.fit_transform_oof(train_df["venue_clean"], y, skf)

    doi_domain_te = TargetEncoder()
    doi_domain_te_oof = doi_domain_te.fit_transform_oof(train_df["doi_domain"], y, skf)

    doi_prefix_te = TargetEncoder()
    doi_prefix_te_oof = doi_prefix_te.fit_transform_oof(train_df["doi_prefix"], y, skf)

    venue_freq = train_df["venue_clean"].value_counts().to_dict()
    doi_domain_freq = train_df["doi_domain"].value_counts().to_dict()
    doi_prefix_freq = train_df["doi_prefix"].value_counts().to_dict()

    author_freq: Dict[str, int] = {}
    for authors in train_df["authors_list"]:
        for a in authors:
            author_freq[a] = author_freq.get(a, 0) + 1

    encoders = {
        "venue_te": venue_te, "doi_domain_te": doi_domain_te, "doi_prefix_te": doi_prefix_te,
        "venue_freq": venue_freq, "doi_domain_freq": doi_domain_freq,
        "doi_prefix_freq": doi_prefix_freq, "author_freq": author_freq,
    }
    train_te = {
        "venue_te": venue_te_oof, "doi_domain_te": doi_domain_te_oof, "doi_prefix_te": doi_prefix_te_oof,
    }
    return encoders, train_te

def build_structured_features(df, encoders, coauthor_degrees, train_te=None):
    venue_freq = df["venue_clean"].map(encoders["venue_freq"]).fillna(0).astype(float)
    doi_domain_freq = df["doi_domain"].map(encoders["doi_domain_freq"]).fillna(0).astype(float)
    doi_prefix_freq = df["doi_prefix"].map(encoders["doi_prefix_freq"]).fillna(0).astype(float)

    if train_te is None:
        venue_te = encoders["venue_te"].transform(df["venue_clean"])
        doi_domain_te = encoders["doi_domain_te"].transform(df["doi_domain"])
        doi_prefix_te = encoders["doi_prefix_te"].transform(df["doi_prefix"])
    else:
        venue_te = train_te["venue_te"]
        doi_domain_te = train_te["doi_domain_te"]
        doi_prefix_te = train_te["doi_prefix_te"]

    year = df["year"].fillna(df["year"].median()).astype(float)

    num_authors, author_freq_mean, author_freq_max, author_freq_min, author_deg_mean, author_deg_max = [], [], [], [], [], []
    author_freq = encoders["author_freq"]

    for authors in df["authors_list"]:
        if not authors:
            num_authors.append(0); author_freq_mean.append(0.0); author_freq_max.append(0.0)
            author_freq_min.append(0.0); author_deg_mean.append(0.0); author_deg_max.append(0.0)
            continue
        freqs = [author_freq.get(a, 0) for a in authors]
        degrees = [coauthor_degrees.get(a, 0) for a in authors]
        num_authors.append(len(authors))
        author_freq_mean.append(float(np.mean(freqs)))
        author_freq_max.append(float(np.max(freqs)))
        author_freq_min.append(float(np.min(freqs)))
        author_deg_mean.append(float(np.mean(degrees)))
        author_deg_max.append(float(np.max(degrees)))

    title_len = df["title_clean"].apply(len).astype(float)
    title_words = df["title_clean"].apply(lambda x: len(x.split())).astype(float)

    features = np.column_stack([
        year.values, title_len.values, title_words.values, venue_freq.values,
        doi_domain_freq.values, doi_prefix_freq.values, venue_te, doi_domain_te,
        doi_prefix_te, df["doi_is_url"].values.astype(float), df["doi_has_doi"].values.astype(float),
        df["doi_len"].values.astype(float), np.array(num_authors, dtype=float),
        np.array(author_freq_mean, dtype=float), np.array(author_freq_max, dtype=float),
        np.array(author_freq_min, dtype=float), np.array(author_deg_mean, dtype=float),
        np.array(author_deg_max, dtype=float),
    ])
    names = ["year", "title_len", "title_words", "venue_freq", "doi_domain_freq", "doi_prefix_freq", "venue_te", "doi_domain_te", "doi_prefix_te", "doi_is_url", "doi_has_doi", "doi_len", "num_authors", "author_freq_mean", "author_freq_max", "author_freq_min", "author_deg_mean", "author_deg_max"]
    return features, names

In [ ]:
def compute_embeddings_with_transformers(texts: List[str], model_name: str, cfg: Config) -> np.ndarray:
    import torch
    from transformers import AutoModel, AutoTokenizer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
    model.to(device)
    model.eval()

    batch_size = 32
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=cfg.embedding_max_length, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1)
            summed = (out.last_hidden_state * mask).sum(dim=1)
            denom = mask.sum(dim=1).clamp(min=1e-6)
            pooled = summed / denom
        outputs.append(pooled.cpu().numpy())
    return np.vstack(outputs)

def build_embedding_cache_path(artifacts_dir: str, prefix: str, model_name: str) -> str:
    model_hash = hashlib.sha256(model_name.encode("utf-8")).hexdigest()[:8]
    safe = re.sub(r"[^a-zA-Z0-9]+", "_", model_name).strip("_")
    filename = f"{prefix}_{safe}_{model_hash}.npy"
    return os.path.join(artifacts_dir, filename)

def load_or_compute_embeddings(texts: List[str], cfg: Config, cache_prefix: str) -> np.ndarray:
    cache_path = build_embedding_cache_path(cfg.artifacts_dir, cache_prefix, cfg.embedding_model_name)
    if os.path.exists(cache_path):
        return np.load(cache_path)

    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(cfg.embedding_model_name)
        embeddings = np.asarray(model.encode(texts, batch_size=64, show_progress_bar=True), dtype=float)
    except Exception as exc:
        fallback_name = "allenai/specter2_base" if cfg.embedding_model_name == "allenai/specter2" else cfg.embedding_model_name
        print(f"SentenceTransformer failed. Falling back to transformers mean pooling with {fallback_name}.")
        embeddings = compute_embeddings_with_transformers(texts, fallback_name, cfg)
        cache_path = build_embedding_cache_path(cfg.artifacts_dir, cache_prefix, fallback_name)

    np.save(cache_path, embeddings)
    return embeddings

def build_feature_matrices(train_df, public_df, private_df, cfg):
    combined = pd.concat([train_df, public_df, private_df], ignore_index=True)
    coauthor_degrees = build_coauthor_degrees(combined["authors_list"])
    author_blocks_train, author_blocks_public, author_blocks_private = [], [], []

    if cfg.author_graph_mode in ("coauthor", "both"):
        coauthor_graph = build_coauthor_graph(combined["authors_list"])
        coauthor_vectors = train_node2vec_embeddings(coauthor_graph, cfg)
        author_blocks_train.append(build_author_embedding_block(train_df["authors_list"], coauthor_vectors, cfg.author_emb_dim, cfg.author_emb_stats))
        author_blocks_public.append(build_author_embedding_block(public_df["authors_list"], coauthor_vectors, cfg.author_emb_dim, cfg.author_emb_stats))
        author_blocks_private.append(build_author_embedding_block(private_df["authors_list"], coauthor_vectors, cfg.author_emb_dim, cfg.author_emb_stats))

    if cfg.author_graph_mode in ("bipartite", "both"):
        combined_ids = combined["id"].astype(str).tolist()
        bipartite_graph = build_bipartite_graph(combined["authors_list"], combined_ids)
        bipartite_vectors = train_node2vec_embeddings(bipartite_graph, cfg)
        author_blocks_train.append(build_author_embedding_block(train_df["authors_list"], bipartite_vectors, cfg.author_emb_dim, cfg.author_emb_stats))
        author_blocks_public.append(build_author_embedding_block(public_df["authors_list"], bipartite_vectors, cfg.author_emb_dim, cfg.author_emb_stats))
        author_blocks_private.append(build_author_embedding_block(private_df["authors_list"], bipartite_vectors, cfg.author_emb_dim, cfg.author_emb_stats))

    y_raw = train_df["Label"].values
    classes = sorted(np.unique(y_raw))
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y = np.array([class_to_idx[c] for c in y_raw], dtype=int)

    encoders, train_te = fit_structured_encoders(train_df, y, cfg)
    train_struct, _ = build_structured_features(train_df, encoders, coauthor_degrees, train_te=train_te)
    public_struct, _ = build_structured_features(public_df, encoders, coauthor_degrees, train_te=None)
    private_struct, _ = build_structured_features(private_df, encoders, coauthor_degrees, train_te=None)

    if author_blocks_train:
        train_struct = np.hstack([train_struct] + author_blocks_train)
        public_struct = np.hstack([public_struct] + author_blocks_public)
        private_struct = np.hstack([private_struct] + author_blocks_private)

    scaler = StandardScaler()
    train_struct = scaler.fit_transform(train_struct)
    public_struct = scaler.transform(public_struct)
    private_struct = scaler.transform(private_struct)

    train_emb = load_or_compute_embeddings(train_df["title_clean"].tolist(), cfg, "emb_train")
    public_emb = load_or_compute_embeddings(public_df["title_clean"].tolist(), cfg, "emb_public")
    private_emb = load_or_compute_embeddings(private_df["title_clean"].tolist(), cfg, "emb_private")

    train_emb = train_emb / (np.linalg.norm(train_emb, axis=1, keepdims=True) + 1e-8)
    public_emb = public_emb / (np.linalg.norm(public_emb, axis=1, keepdims=True) + 1e-8)
    private_emb = private_emb / (np.linalg.norm(private_emb, axis=1, keepdims=True) + 1e-8)

    if cfg.embedding_pca_dim and train_emb.shape[1] > cfg.embedding_pca_dim:
        pca = PCA(n_components=cfg.embedding_pca_dim, random_state=cfg.random_seed)
        train_emb = pca.fit_transform(train_emb)
        public_emb = pca.transform(public_emb)
        private_emb = pca.transform(private_emb)
        train_emb = train_emb / (np.linalg.norm(train_emb, axis=1, keepdims=True) + 1e-8)
        public_emb = public_emb / (np.linalg.norm(public_emb, axis=1, keepdims=True) + 1e-8)
        private_emb = private_emb / (np.linalg.norm(private_emb, axis=1, keepdims=True) + 1e-8)

    x_train = np.hstack([train_emb, train_struct])
    x_public = np.hstack([public_emb, public_struct])
    x_private = np.hstack([private_emb, private_struct])

    return x_train, x_public, x_private, y, classes

In [ ]:
def build_gbdt_model(cfg: Config):
    try:
        import lightgbm as lgb
        return lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, max_depth=-1, num_leaves=64, subsample=0.9, colsample_bytree=0.9, random_state=cfg.random_seed)
    except Exception:
        import xgboost as xgb
        return xgb.XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=8, subsample=0.9, colsample_bytree=0.9, objective="reg:squarederror", random_state=cfg.random_seed, n_jobs=-1)

def train_gbdt_cv(x, y, x_test, cfg):
    skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_seed)
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(len(x_test), dtype=float)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(x, y), start=1):
        model = build_gbdt_model(cfg)
        model.fit(x[tr_idx], y[tr_idx])
        oof[val_idx] = model.predict(x[val_idx])
        test_preds += model.predict(x_test) / cfg.n_splits
        print(f"[GBDT] fold={fold} done")
    return oof, test_preds

def build_transformer_text(df):
    texts = []
    for title, venue, year in zip(df["title_clean"], df["venue_clean"], df["year"]):
        parts = [title]
        if venue and venue != "<missing>": parts.append(f"venue {venue}")
        if not (isinstance(year, float) and np.isnan(year)): parts.append(f"year {int(year)}")
        texts.append(" . ".join(parts))
    return texts

def build_tfidf_text(df):
    texts = []
    for title, venue, year, authors in zip(df["title_clean"], df["venue_clean"], df["year"], df["authors_list"]):
        parts = [title]
        if venue and venue != "<missing>": parts.append(f"venue {venue}")
        if authors: parts.append("authors " + " ".join(authors))
        if not (isinstance(year, float) and np.isnan(year)): parts.append(f"year {int(year)}")
        texts.append(" . ".join(parts))
    return texts

def train_tfidf_cv(train_texts, y, test_texts, cfg):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import Ridge
    from scipy.sparse import hstack

    word_vec = TfidfVectorizer(ngram_range=cfg.tfidf_word_ngram, min_df=cfg.tfidf_min_df, max_df=cfg.tfidf_max_df, max_features=cfg.tfidf_word_max_features)
    char_vec = TfidfVectorizer(analyzer="char", ngram_range=cfg.tfidf_char_ngram, min_df=cfg.tfidf_min_df, max_df=cfg.tfidf_max_df, max_features=cfg.tfidf_char_max_features)

    x_train = hstack([word_vec.fit_transform(train_texts), char_vec.fit_transform(train_texts)]).tocsr()
    x_test = hstack([word_vec.transform(test_texts), char_vec.transform(test_texts)]).tocsr()

    skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_seed)
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(len(test_texts), dtype=float)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(x_train, y), start=1):
        model = Ridge(alpha=cfg.tfidf_alpha)
        model.fit(x_train[tr_idx], y[tr_idx])
        oof[val_idx] = model.predict(x_train[val_idx])
        test_preds += model.predict(x_test) / cfg.n_splits
        print(f"[TF-IDF] fold={fold} done")
    return oof, test_preds

def train_transformer_cv(train_texts, y, test_texts, cfg):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
    class TextRegDataset(torch.utils.data.Dataset):
        def __init__(self, texts, labels, tokenizer):
            self.texts, self.labels, self.tokenizer = texts, labels, tokenizer
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=cfg.transformer_max_length, return_tensors="pt")
            item = {k: v.squeeze(0) for k, v in enc.items()}
            if self.labels is not None: item["labels"] = torch.tensor(float(self.labels[idx]))
            return item

    tokenizer = AutoTokenizer.from_pretrained(cfg.transformer_model_name)
    skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_seed)
    oof = np.zeros(len(y), dtype=float)
    test_preds = np.zeros(len(test_texts), dtype=float)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(train_texts, y), start=1):
        model = AutoModelForSequenceClassification.from_pretrained(cfg.transformer_model_name, num_labels=1, problem_type="regression")
        train_ds = TextRegDataset([train_texts[i] for i in tr_idx], y[tr_idx], tokenizer)
        val_ds = TextRegDataset([train_texts[i] for i in val_idx], y[val_idx], tokenizer)
        test_ds = TextRegDataset(test_texts, None, tokenizer)

        args = TrainingArguments(
            output_dir=os.path.join(cfg.artifacts_dir, f"transformer_fold_{fold}"),
            num_train_epochs=cfg.transformer_epochs, learning_rate=cfg.transformer_lr,
            per_device_train_batch_size=cfg.transformer_batch_size, per_device_eval_batch_size=cfg.transformer_batch_size,
            evaluation_strategy="epoch", save_strategy="no", logging_steps=50, report_to=[],
            fp16=cfg.use_fp16 and torch.cuda.is_available()
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
        trainer.train()
        oof[val_idx] = trainer.predict(val_ds).predictions.squeeze()
        test_preds += trainer.predict(test_ds).predictions.squeeze() / cfg.n_splits
        print(f"[Transformer] fold={fold} done")
    return oof, test_preds

In [ ]:
def optimize_blend_weight(preds_a, preds_b, y, step=0.05):
    best_w, best_rmse = 0.5, float("inf")
    for w in np.arange(0.0, 1.0 + 1e-9, step):
        blended = w * preds_a + (1.0 - w) * preds_b
        score = rmse(y, blended)
        if score < best_rmse: best_rmse, best_w = score, float(w)
    return best_w, best_rmse

def optimize_blend_weights(preds_list, y, step=0.05):
    if len(preds_list) == 1: return [1.0], rmse(y, preds_list[0])
    if len(preds_list) == 2:
        w, score = optimize_blend_weight(preds_list[0], preds_list[1], y, step)
        return [w, 1.0 - w], score
    best_weights, best_score = [1.0, 0.0, 0.0], float("inf")
    grid = np.arange(0.0, 1.0 + 1e-9, step)
    for w1 in grid:
        for w2 in grid:
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9: continue
            if w3 < 0.0: w3 = 0.0
            blended = w1 * preds_list[0] + w2 * preds_list[1] + w3 * preds_list[2]
            score = rmse(y, blended)
            if score < best_score:
                best_score = score
                best_weights = [float(w1), float(w2), float(w3)]
    return best_weights, best_score

def save_predictions(ids: np.ndarray, labels: np.ndarray, path: str) -> None:
    pd.DataFrame({"id": ids, "Label": labels}).to_csv(path, index=False)

In [ ]:
!pip install node2vec

In [ ]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall numpy

In [ ]:
# 1. Setup Data & Thư mục
set_seed(cfg.random_seed)
ensure_dir(cfg.output_dir)
ensure_dir(cfg.artifacts_dir)
tracker = ExperimentTracker(cfg.artifacts_dir)

train_df, public_df, private_df = load_data(cfg)
save_json(os.path.join(cfg.artifacts_dir, "eda_summary.json"), eda_summary(train_df))

print("Tiến hành preprocess dữ liệu...")
train_df = clean_noise(train_df)
train_df = preprocess_dataframe(train_df)
public_df = preprocess_dataframe(public_df)
private_df = preprocess_dataframe(private_df)

print("Tạo ma trận Features (có thể mất thời gian)...")
x_train, x_public, x_private, y, classes = build_feature_matrices(train_df, public_df, private_df, cfg)

# 2. Huấn luyện Model 1: GBDT
print("Bắt đầu train GBDT...")
x_test_all = np.vstack([x_public, x_private])
gbdt_oof, gbdt_test_preds = train_gbdt_cv(x_train, y, x_test_all, cfg)
gbdt_public_pred = gbdt_test_preds[: len(public_df)]
gbdt_private_pred = gbdt_test_preds[len(public_df) :]
gbdt_qwk = qwk(y, apply_thresholds(gbdt_oof, optimize_thresholds(gbdt_oof, y, len(classes))))

# 3. Huấn luyện Model 2: TF-IDF
print("Bắt đầu train TF-IDF...")
tfidf_train_texts = build_tfidf_text(train_df)
tfidf_test_texts = build_tfidf_text(public_df) + build_tfidf_text(private_df)
tfidf_oof, tfidf_test_preds = train_tfidf_cv(tfidf_train_texts, y, tfidf_test_texts, cfg)
tfidf_public_pred = tfidf_test_preds[: len(public_df)]
tfidf_private_pred = tfidf_test_preds[len(public_df) :]
tfidf_qwk = qwk(y, apply_thresholds(tfidf_oof, optimize_thresholds(tfidf_oof, y, len(classes))))

# 4. Huấn luyện Model 3: Transformer (Tùy chọn/Sẽ bị bỏ qua nếu thiếu libs/GPU config nặng)
transformer_oof, transformer_public_pred, transformer_private_pred, transformer_qwk = None, None, None, None
try:
    print("Bắt đầu train Transformer...")
    test_texts = build_transformer_text(public_df) + build_transformer_text(private_df)
    transformer_oof, transformer_test_preds = train_transformer_cv(build_transformer_text(train_df), y, test_texts, cfg)
    transformer_public_pred = transformer_test_preds[: len(public_df)]
    transformer_private_pred = transformer_test_preds[len(public_df) :]
    transformer_qwk = qwk(y, apply_thresholds(transformer_oof, optimize_thresholds(transformer_oof, y, len(classes))))
except Exception as exc:
    print(f"Bỏ qua Transformer: {exc}")

# 5. Ensemble & Lưu kết quả
models = [("gbdt", gbdt_oof, gbdt_public_pred, gbdt_private_pred), ("tfidf", tfidf_oof, tfidf_public_pred, tfidf_private_pred)]
if transformer_oof is not None: models.append(("transformer", transformer_oof, transformer_public_pred, transformer_private_pred))

blend_weights, blend_rmse = optimize_blend_weights([m[1] for m in models], y)
oof_blend, public_blend, private_blend = np.zeros(len(y)), np.zeros(len(public_df)), np.zeros(len(private_df))
for weight, model in zip(blend_weights, models):
    oof_blend += weight * model[1]
    public_blend += weight * model[2]
    private_blend += weight * model[3]

final_thr = optimize_thresholds(oof_blend, y, len(classes))
ensemble_qwk = qwk(y, apply_thresholds(oof_blend, final_thr))

combined_ids = np.concatenate([public_df["id"].values, private_df["id"].values])
combined_labels = np.concatenate([np.array(classes)[apply_thresholds(public_blend, final_thr)], np.array(classes)[apply_thresholds(private_blend, final_thr)]])
save_predictions(combined_ids, combined_labels, os.path.join(cfg.output_dir, "submission.csv"))

tracker.log_run(
    params={"embedding_model": cfg.embedding_model_name, "transformer_model": cfg.transformer_model_name},
    metrics={"qwk_gbdt": gbdt_qwk, "qwk_tfidf": tfidf_qwk, "qwk_ensemble": ensemble_qwk, "rmse_blend": blend_rmse},
    tags={"blend_weights": blend_weights}
)
print("Hoàn tất! Kết quả đã được lưu tại outputs/submission.csv.")